# Tornado activity and county vulnerability

Disclaimer: The code generated in this notebook was generated in collaboration with CHAT-GPT 6.

In [ ]:
from pathlib import Path

# Find the shared repository data directory.
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / "data" / "vulnerability_analysis").is_dir():
        ANALYSIS_DIR = directory
        break
else:
    raise FileNotFoundError("Run this notebook from the repository root or a subfolder.")

RAW_FILES_DIR = ANALYSIS_DIR / "data" / "vulnerability_analysis"
DATASETS_DIR = RAW_FILES_DIR

import numpy as np
import pandas as pd

In [ ]:
required_files = [
    DATASETS_DIR / "tornados50-26.csv",
    DATASETS_DIR / "counties.geojson",
    *(RAW_FILES_DIR / name for name in [
        "est24all.csv", "ACSST5Y2024.S2701-Data.csv", "cars.csv",
        "diability.csv", "age.csv", "Hospital_General_Information.csv",
        "Ruralurbancontinuumcodes2023.csv",
    ]),
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing local input files:\n" + "\n".join(missing))
print(f"Using local data from {ANALYSIS_DIR}")

## Load downloaded tornado events

Use the saved 1950–2025 events directly. Ignore index columns left by earlier CSV exports and retain zero-padded county identifiers.

In [ ]:
tornados = pd.read_csv(
    DATASETS_DIR / "tornados50-26.csv",
    dtype={"STATE_FIPS": "string", "CZ_FIPS": "string", "COUNTY_FIPS": "string"},
    usecols=lambda column: not column.startswith("Unnamed:"),
)
tornados["COUNTY_FIPS"] = (
    tornados["STATE_FIPS"].str.zfill(2)
    + tornados["CZ_FIPS"].str.zfill(3)
)
print(f"Loaded {len(tornados):,} tornado events")
tornados.head()

## Prepare county poverty and income data

In [ ]:
df = pd.read_csv(RAW_FILES_DIR / "est24all.csv", header=[2, 3])

In [ ]:
df.head()

In [ ]:
print(df.shape)

In [ ]:
print(df.info())

In [ ]:
econ = df[[
    ("Geographic Information", "State FIPS Code"),
    ("Unnamed: 1_level_0", "County FIPS Code"),
    ("Unnamed: 3_level_0", "Name"),
    ("Unnamed: 7_level_0", "Poverty Percent, All Ages"),
    ("Median Household Income", "Median Household Income")
]].copy()

In [ ]:
econ.columns = [
    "STATE_FIPS",
    "COUNTY_FIPS_PART",
    "COUNTY_NAME",
    "POVERTY_RATE",
    "MEDIAN_HOUSEHOLD_INCOME"
]

In [ ]:
econ.head()

In [ ]:
econ["COUNTY_FIPS"] = econ["STATE_FIPS"].astype(str).str.zfill(2) + econ["COUNTY_FIPS_PART"].astype(str).str.zfill(3)

In [ ]:
econ.head()

In [ ]:
econ["POVERTY_RATE"] = pd.to_numeric(econ["POVERTY_RATE"], errors='coerce')

econ["MEDIAN_HOUSEHOLD_INCOME"] = (econ["MEDIAN_HOUSEHOLD_INCOME"].astype(str).str.replace(",", "", regex=False))
econ["MEDIAN_HOUSEHOLD_INCOME"] = pd.to_numeric(econ["MEDIAN_HOUSEHOLD_INCOME"], errors="coerce")

In [ ]:
econ = econ.drop(
    columns=["STATE_FIPS", "COUNTY_FIPS_PART"]
)

In [ ]:
print(econ.columns)

In [ ]:
data = pd.merge(tornados, econ, on=["COUNTY_FIPS"], how='left')

In [ ]:
data.info()

## Aggregate county events and compare early and late periods

In [ ]:
keep = [
    "EVENT_ID",
    "YEAR",
    "STATE",
    "COUNTY_FIPS",
    "COUNTY_NAME",
    "BEGIN_LAT",
    "BEGIN_LON",
    "TOR_F_SCALE",
    "TOR_LENGTH",
    "TOR_WIDTH",
    "INJURIES_DIRECT",
    "DEATHS_DIRECT",
    "DAMAGE_PROPERTY",
    "DAMAGE_CROPS",
    "POVERTY_RATE",
    "MEDIAN_HOUSEHOLD_INCOME"
]

merged = data[keep].copy()

dataset = (
    merged.groupby("COUNTY_FIPS")
    .agg(
        COUNTY_NAME=("COUNTY_NAME", "first"),
        STATE=("STATE", "first"),
        POVERTY_RATE=("POVERTY_RATE", "first"),
        MEDIAN_HOUSEHOLD_INCOME=("MEDIAN_HOUSEHOLD_INCOME", "first"),
        tornado_count=("EVENT_ID", "count"),
        total_path_length=("TOR_LENGTH", "sum"),
        avg_width=("TOR_WIDTH", "mean"),
        injuries=("INJURIES_DIRECT", "sum"),
        deaths=("DEATHS_DIRECT", "sum"),
        mean_lat=("BEGIN_LAT", "mean"),
        mean_lon=("BEGIN_LON", "mean")
    )
    .reset_index()
)

# F/EF1+
ef1plus = [
    "F1", "F2", "F3", "F4", "F5",
    "EF1", "EF2", "EF3", "EF4", "EF5"
]

strong1 = merged[merged["TOR_F_SCALE"].isin(ef1plus)]

trend1 = pd.concat([
    strong1[strong1["YEAR"] <= 1987]
        .groupby("COUNTY_FIPS").size().rename("EF1plus_early"),

    strong1[strong1["YEAR"] >= 1988]
        .groupby("COUNTY_FIPS").size().rename("EF1plus_late")
], axis=1).fillna(0)

trend1["EF1plus_change"] = (
    trend1["EF1plus_late"] - trend1["EF1plus_early"]
)

# F/EF2+
ef2plus = [
    "F2", "F3", "F4", "F5",
    "EF2", "EF3", "EF4", "EF5"
]

strong2 = merged[merged["TOR_F_SCALE"].isin(ef2plus)]

trend2 = pd.concat([
    strong2[strong2["YEAR"] <= 1987]
        .groupby("COUNTY_FIPS").size().rename("EF2plus_early"),

    strong2[strong2["YEAR"] >= 1988]
        .groupby("COUNTY_FIPS").size().rename("EF2plus_late")
], axis=1).fillna(0)

trend2["EF2plus_change"] = (
    trend2["EF2plus_late"] - trend2["EF2plus_early"]
)

# Final county-level analysis dataset
dataset = (
    dataset
    .merge(trend1, on="COUNTY_FIPS", how="left")
    .merge(trend2, on="COUNTY_FIPS", how="left")
)

trend_cols = [
    "EF1plus_early",
    "EF1plus_late",
    "EF1plus_change",
    "EF2plus_early",
    "EF2plus_late",
    "EF2plus_change"
]

dataset[trend_cols] = dataset[trend_cols].fillna(0)

In [ ]:
dataset.head()

## Map tornado activity and poverty using local county boundaries

In [ ]:
import geopandas as gpd

counties = gpd.read_file(
    DATASETS_DIR / "counties.geojson"
)

counties["COUNTY_FIPS"] = counties["id"].astype(str).str.zfill(5)

map_df = counties.merge(
    dataset,
    on="COUNTY_FIPS",
    how="left"
)

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

v = max(
    abs(map_df["EF1plus_change"].min()),
    abs(map_df["EF1plus_change"].max())
)

fig, ax = plt.subplots(figsize=(14, 8))

map_df.plot(
    column="EF1plus_change",
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-v, vcenter=0, vmax=v),
    linewidth=0.15,
    edgecolor="white",
    legend=True,
    ax=ax
)

ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_axis_off()
ax.set_title("Change in F/EF1+ Tornado Activity: 1950–1987 vs. 1988–2025")

plt.show()

In [ ]:
# Same map for F/EF2+
v = max(
    abs(map_df["EF2plus_change"].min()),
    abs(map_df["EF2plus_change"].max())
)

fig, ax = plt.subplots(figsize=(14, 8))

map_df.plot(
    column="EF2plus_change",
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-v, vcenter=0, vmax=v),
    linewidth=0.15,
    edgecolor="white",
    legend=True,
    ax=ax
)

ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_axis_off()
ax.set_title("Change in F/EF2+ Tornado Activity: 1950–1987 vs. 1988–2025")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

map_df.plot(
    column="POVERTY_RATE",
    cmap="OrRd",
    legend=True,
    linewidth=0.1,
    edgecolor="white",
    ax=ax
)

ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_axis_off()
ax.set_title("County Poverty Rate")

plt.show()

## Compare poverty and income across tornado trends

In [ ]:
comparison = (
    dataset
    .assign(
        TREND_GROUP=np.where(
            dataset["EF1plus_change"] > 0,
            "Increasing",
            np.where(dataset["EF1plus_change"] < 0, "Decreasing", "No change")
        )
    )
    .groupby("TREND_GROUP")[
        ["POVERTY_RATE", "MEDIAN_HOUSEHOLD_INCOME"]
    ]
    .mean()
)

comparison

In [ ]:
dataset.assign(
    TREND_GROUP=np.where(
        dataset["EF1plus_change"] > 0,
        "Increasing",
        np.where(dataset["EF1plus_change"] < 0, "Decreasing", "No change")
    )
)["TREND_GROUP"].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plot_df = dataset.copy()

plot_df["TREND_GROUP"] = np.where(
    plot_df["EF1plus_change"] > 0,
    "Increasing",
    np.where(plot_df["EF1plus_change"] < 0, "Decreasing", "No change")
)

sns.boxplot(
    data=plot_df,
    x="TREND_GROUP",
    y="POVERTY_RATE"
)

plt.xlabel("")
plt.ylabel("Poverty rate (%)")
plt.title("County Poverty Rate by F/EF1+ Tornado Trend")
plt.show()

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    "POVERTY_RATE ~ C(TREND_GROUP) + mean_lat + mean_lon",
    data=plot_df
).fit()

print(model.summary())

## Prepare insurance coverage and compare county trends

In [ ]:
df = pd.read_csv(RAW_FILES_DIR / "ACSST5Y2024.S2701-Data.csv")

In [ ]:
df.shape

In [ ]:
for i, col in enumerate(df.columns):
    print(i, col)

In [ ]:
uninsured = df[
    ["GEO_ID", "NAME", "S2701_C05_001E"]
].copy()

uninsured.columns = [
    "GEO_ID",
    "COUNTY_NAME_INSURANCE",
    "PERCENT_UNINSURED"
]

In [ ]:
uninsured.head()

In [ ]:
uninsured = uninsured.iloc[1:].copy()

uninsured["PERCENT_UNINSURED"] = pd.to_numeric(
    uninsured["PERCENT_UNINSURED"],
    errors="coerce"
)

uninsured["COUNTY_FIPS"] = uninsured["GEO_ID"].str[-5:]

In [ ]:
uninsured.head()

In [ ]:
dataset = dataset.merge(
    uninsured[["COUNTY_FIPS", "PERCENT_UNINSURED"]],
    on="COUNTY_FIPS",
    how="left"
)

In [ ]:
dataset.head()

In [ ]:
dataset["PERCENT_UNINSURED"].isna().sum()

In [ ]:
dataset.loc[
    dataset["PERCENT_UNINSURED"].isna(),
    ["COUNTY_FIPS", "COUNTY_NAME", "STATE"]
]

In [ ]:
# keep only standard 5-digit county/county-equivalent FIPS
dataset = dataset[
    ~dataset["COUNTY_FIPS"].str.endswith("000")
].copy()

In [ ]:
dataset.loc[
    dataset["PERCENT_UNINSURED"].isna(),
    ["COUNTY_FIPS", "COUNTY_NAME", "STATE"]
]

In [ ]:
dataset = dataset.dropna(
    subset=["PERCENT_UNINSURED"]
).copy()

In [ ]:
dataset["PERCENT_UNINSURED"].isna().sum()

In [ ]:
import numpy as np

dataset["TREND_GROUP"] = np.where(
    dataset["EF1plus_change"] > 0,
    "Increasing",
    np.where(
        dataset["EF1plus_change"] < 0,
        "Decreasing",
        "No change"
    )
)

In [ ]:
summary = (
    dataset.groupby("TREND_GROUP")[
        [
            "POVERTY_RATE",
            "MEDIAN_HOUSEHOLD_INCOME",
            "PERCENT_UNINSURED"
        ]
    ]
    .agg(["mean", "median"])
)

summary

In [ ]:
map_df = counties.merge(
    dataset,
    on="COUNTY_FIPS",
    how="left"
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 8))

map_df.plot(
    column="PERCENT_UNINSURED",
    cmap="OrRd",
    legend=True,
    linewidth=0.1,
    edgecolor="white",
    ax=ax
)

ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_axis_off()
ax.set_title("Percent Uninsured by County")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

order = ["Decreasing", "No change", "Increasing"]

data_to_plot = [
    dataset.loc[
        dataset["TREND_GROUP"] == group,
        "PERCENT_UNINSURED"
    ].dropna()
    for group in order
]

plt.figure(figsize=(8, 6))

plt.boxplot(
    data_to_plot,
    tick_labels=order,
    showfliers=False
)

plt.ylabel("Percent uninsured")
plt.xlabel("F/EF1+ tornado trend")
plt.title("Uninsured Rate by Tornado Activity Trend")

plt.show()

In [ ]:
group_summary = (
    dataset.groupby("TREND_GROUP")
    .agg(
        n=("COUNTY_FIPS", "count"),
        poverty_mean=("POVERTY_RATE", "mean"),
        poverty_median=("POVERTY_RATE", "median"),
        income_mean=("MEDIAN_HOUSEHOLD_INCOME", "mean"),
        income_median=("MEDIAN_HOUSEHOLD_INCOME", "median"),
        uninsured_mean=("PERCENT_UNINSURED", "mean"),
        uninsured_median=("PERCENT_UNINSURED", "median")
    )
)

group_summary

In [ ]:
from scipy.stats import ttest_ind

inc = dataset.loc[
    dataset["TREND_GROUP"] == "Increasing",
    "PERCENT_UNINSURED"
].dropna()

dec = dataset.loc[
    dataset["TREND_GROUP"] == "Decreasing",
    "PERCENT_UNINSURED"
].dropna()

ttest_ind(
    inc,
    dec,
    equal_var=False
)

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt(
        ((len(a) - 1) * a.var() + (len(b) - 1) * b.var())
        / (len(a) + len(b) - 2)
    )

    return (a.mean() - b.mean()) / pooled_sd

cohens_d(inc, dec)

In [ ]:
high_increase_threshold = dataset["EF1plus_change"].quantile(0.75)

high_increase = dataset[
    dataset["EF1plus_change"] >= high_increase_threshold
].copy()

In [ ]:
high_increase.sort_values(
    "PERCENT_UNINSURED",
    ascending=False
)[
    [
        "COUNTY_NAME",
        "STATE",
        "EF1plus_change",
        "POVERTY_RATE",
        "MEDIAN_HOUSEHOLD_INCOME",
        "PERCENT_UNINSURED"
    ]
].head(20)

In [ ]:
print("Increasing mean:", inc.mean())
print("Decreasing mean:", dec.mean())
print("Difference:", inc.mean() - dec.mean())
print("Cohen's d:", cohens_d(inc, dec))

## Vehicle access

In [ ]:
vehicles = pd.read_csv(RAW_FILES_DIR / "cars.csv")

In [ ]:
print(vehicles.columns)

In [ ]:
vehicles = vehicles[
    ["GEO_ID", "NAME", "B08201_001E", "B08201_002E"]
].copy()

vehicles.columns = [
    "GEO_ID",
    "COUNTY_NAME_VEHICLE",
    "TOTAL_HOUSEHOLDS",
    "NO_VEHICLE_HOUSEHOLDS"
]

vehicles = vehicles.iloc[1:].copy()

vehicles["TOTAL_HOUSEHOLDS"] = pd.to_numeric(
    vehicles["TOTAL_HOUSEHOLDS"],
    errors="coerce"
)

vehicles["NO_VEHICLE_HOUSEHOLDS"] = pd.to_numeric(
    vehicles["NO_VEHICLE_HOUSEHOLDS"],
    errors="coerce"
)

vehicles["COUNTY_FIPS"] = vehicles["GEO_ID"].str[-5:]

vehicles["NO_VEHICLE_RATE"] = (
    vehicles["NO_VEHICLE_HOUSEHOLDS"]
    / vehicles["TOTAL_HOUSEHOLDS"]
    * 100
)

dataset = dataset.merge(
    vehicles[["COUNTY_FIPS", "NO_VEHICLE_RATE"]],
    on="COUNTY_FIPS",
    how="left"
)

print(dataset["NO_VEHICLE_RATE"].isna().sum())

In [ ]:
from scipy.stats import ttest_ind
import numpy as np

inc_vehicle = dataset.loc[
    dataset["TREND_GROUP"] == "Increasing",
    "NO_VEHICLE_RATE"
].dropna()

dec_vehicle = dataset.loc[
    dataset["TREND_GROUP"] == "Decreasing",
    "NO_VEHICLE_RATE"
].dropna()

# Group means
print("Increasing mean:", inc_vehicle.mean())
print("Decreasing mean:", dec_vehicle.mean())
print("Difference:", inc_vehicle.mean() - dec_vehicle.mean())

# Welch t-test
print(
    ttest_ind(
        inc_vehicle,
        dec_vehicle,
        equal_var=False
    )
)

# Cohen's d
def cohens_d(a, b):
    pooled_sd = np.sqrt(
        ((len(a) - 1) * a.var() + (len(b) - 1) * b.var())
        / (len(a) + len(b) - 2)
    )
    return (a.mean() - b.mean()) / pooled_sd

print("Cohen's d:", cohens_d(inc_vehicle, dec_vehicle))

In [ ]:
import matplotlib.pyplot as plt

order = ["Decreasing", "No change", "Increasing"]

data_to_plot = [
    dataset.loc[
        dataset["TREND_GROUP"] == group,
        "NO_VEHICLE_RATE"
    ].dropna()
    for group in order
]

plt.figure(figsize=(8, 6))

plt.boxplot(
    data_to_plot,
    tick_labels=order,
    showfliers=False
)

plt.ylabel("Households with no vehicle (%)")
plt.xlabel("F/EF1+ tornado trend")
plt.title("Vehicle Access by Tornado Activity Trend")

plt.show()

In [ ]:
from scipy.stats import pearsonr, spearmanr

test_df = dataset[
    ["EF1plus_change", "NO_VEHICLE_RATE"]
].dropna()

print("Pearson:", pearsonr(
    test_df["EF1plus_change"],
    test_df["NO_VEHICLE_RATE"]
))

print("Spearman:", spearmanr(
    test_df["EF1plus_change"],
    test_df["NO_VEHICLE_RATE"]
))

In [ ]:
print(vehicles.info())

## Disability prevalence

In [ ]:
disability = pd.read_csv(RAW_FILES_DIR / "diability.csv")

In [ ]:
disability.columns.tolist()

In [ ]:
disability = disability[
    ["GEO_ID", "NAME", "S1810_C03_001E"]
].copy()

disability.columns = [
    "GEO_ID",
    "COUNTY_NAME_DISABILITY",
    "DISABILITY_RATE"
]

# Drop Census metadata row
disability = disability.iloc[1:].copy()

# Convert to numeric
disability["DISABILITY_RATE"] = pd.to_numeric(
    disability["DISABILITY_RATE"],
    errors="coerce"
)

# Extract 5-digit county FIPS
disability["COUNTY_FIPS"] = disability["GEO_ID"].str[-5:]

# Merge into main county dataset
dataset = dataset.merge(
    disability[["COUNTY_FIPS", "DISABILITY_RATE"]],
    on="COUNTY_FIPS",
    how="left"
)

print("Missing:", dataset["DISABILITY_RATE"].isna().sum())
dataset[["COUNTY_NAME", "DISABILITY_RATE"]].head()

In [ ]:
print("dataset duplicates:", dataset["COUNTY_FIPS"].duplicated().sum())
print("disability duplicates:", disability["COUNTY_FIPS"].duplicated().sum())

In [ ]:
# Inspect duplicate counties and remove them if they are duplicate county rows

dupes = dataset[
    dataset["COUNTY_FIPS"].duplicated(keep=False)
].sort_values("COUNTY_FIPS")

print("Duplicate county rows:", len(dupes))
print("Duplicate FIPS:", dataset["COUNTY_FIPS"].duplicated().sum())

print("\nHow many rows per duplicated county:")
print(
    dupes.groupby("COUNTY_FIPS")
    .size()
    .value_counts()
)

print("\nMax number of unique values within duplicated counties:")
print(
    dupes.groupby("COUNTY_FIPS")
    .nunique()
    .max()
)

display(
    dupes[
        [
            "COUNTY_FIPS",
            "COUNTY_NAME",
            "STATE",
            "PERCENT_UNINSURED",
            "NO_VEHICLE_RATE",
            "DISABILITY_RATE"
        ]
    ].head(100)
)

# Keep only one row per county
dataset = dataset.drop_duplicates(
    subset="COUNTY_FIPS",
    keep="first"
).copy()

print("\nAfter cleanup:")
print("Duplicate FIPS:", dataset["COUNTY_FIPS"].duplicated().sum())
print("Shape:", dataset.shape)

In [ ]:
from scipy.stats import ttest_ind, pearsonr, spearmanr
import numpy as np

inc_disability = dataset.loc[
    dataset["TREND_GROUP"] == "Increasing",
    "DISABILITY_RATE"
].dropna()

dec_disability = dataset.loc[
    dataset["TREND_GROUP"] == "Decreasing",
    "DISABILITY_RATE"
].dropna()

print("Increasing mean:", inc_disability.mean())
print("Decreasing mean:", dec_disability.mean())
print("Difference:", inc_disability.mean() - dec_disability.mean())

print(
    ttest_ind(
        inc_disability,
        dec_disability,
        equal_var=False
    )
)

def cohens_d(a, b):
    pooled_sd = np.sqrt(
        ((len(a) - 1) * a.var() + (len(b) - 1) * b.var())
        / (len(a) + len(b) - 2)
    )
    return (a.mean() - b.mean()) / pooled_sd

print("Cohen's d:", cohens_d(inc_disability, dec_disability))

test_df = dataset[
    ["EF1plus_change", "DISABILITY_RATE"]
].dropna()

print("Pearson:", pearsonr(
    test_df["EF1plus_change"],
    test_df["DISABILITY_RATE"]
))

print("Spearman:", spearmanr(
    test_df["EF1plus_change"],
    test_df["DISABILITY_RATE"]
))

In [ ]:
from scipy.stats import ttest_ind, pearsonr, spearmanr
import numpy as np

# Create F/EF2+ trend groups
dataset["TREND_GROUP_EF2"] = np.where(
    dataset["EF2plus_change"] > 0,
    "Increasing",
    np.where(
        dataset["EF2plus_change"] < 0,
        "Decreasing",
        "No change"
    )
)

# Increasing vs decreasing disability rates
inc_disability_ef2 = dataset.loc[
    dataset["TREND_GROUP_EF2"] == "Increasing",
    "DISABILITY_RATE"
].dropna()

dec_disability_ef2 = dataset.loc[
    dataset["TREND_GROUP_EF2"] == "Decreasing",
    "DISABILITY_RATE"
].dropna()

print("Increasing mean:", inc_disability_ef2.mean())
print("Decreasing mean:", dec_disability_ef2.mean())
print("Difference:", inc_disability_ef2.mean() - dec_disability_ef2.mean())

print(
    "T-test:",
    ttest_ind(
        inc_disability_ef2,
        dec_disability_ef2,
        equal_var=False
    )
)

# Cohen's d
def cohens_d(a, b):
    pooled_sd = np.sqrt(
        ((len(a) - 1) * a.var() + (len(b) - 1) * b.var())
        / (len(a) + len(b) - 2)
    )
    return (a.mean() - b.mean()) / pooled_sd

print(
    "Cohen's d:",
    cohens_d(
        inc_disability_ef2,
        dec_disability_ef2
    )
)

# Continuous association
test_df_ef2 = dataset[
    ["EF2plus_change", "DISABILITY_RATE"]
].dropna()

print(
    "Pearson:",
    pearsonr(
        test_df_ef2["EF2plus_change"],
        test_df_ef2["DISABILITY_RATE"]
    )
)

print(
    "Spearman:",
    spearmanr(
        test_df_ef2["EF2plus_change"],
        test_df_ef2["DISABILITY_RATE"]
    )
)

## Age vulnerability

In [ ]:
age = pd.read_csv(RAW_FILES_DIR / "age.csv")

In [ ]:
age.columns[:40]

In [ ]:
from scipy.stats import ttest_ind, pearsonr, spearmanr
import numpy as np
import pandas as pd

# Keep age variables
age2 = age[
    ["GEO_ID", "NAME", "S0101_C02_002E", "S0101_C02_030E"]
].copy()

age2.columns = [
    "GEO_ID",
    "COUNTY_NAME_AGE",
    "UNDER_5_RATE",
    "AGE_65_PLUS_RATE"
]

# Drop Census metadata row
age2 = age2.iloc[1:].copy()

# Convert to numeric
age2["UNDER_5_RATE"] = pd.to_numeric(
    age2["UNDER_5_RATE"],
    errors="coerce"
)

age2["AGE_65_PLUS_RATE"] = pd.to_numeric(
    age2["AGE_65_PLUS_RATE"],
    errors="coerce"
)

# Extract FIPS
age2["COUNTY_FIPS"] = age2["GEO_ID"].str[-5:]

# Merge
dataset = dataset.merge(
    age2[
        ["COUNTY_FIPS", "UNDER_5_RATE", "AGE_65_PLUS_RATE"]
    ],
    on="COUNTY_FIPS",
    how="left"
)

print("Missing under 5:", dataset["UNDER_5_RATE"].isna().sum())
print("Missing age 65+:", dataset["AGE_65_PLUS_RATE"].isna().sum())

def cohens_d(a, b):
    pooled_sd = np.sqrt(
        ((len(a) - 1) * a.var() + (len(b) - 1) * b.var())
        / (len(a) + len(b) - 2)
    )
    return (a.mean() - b.mean()) / pooled_sd

def test_variable(var):
    print(f"\n===== {var} =====")

    # F/EF1+
    inc1 = dataset.loc[
        dataset["TREND_GROUP"] == "Increasing",
        var
    ].dropna()

    dec1 = dataset.loc[
        dataset["TREND_GROUP"] == "Decreasing",
        var
    ].dropna()

    print("\nF/EF1+")
    print("Increasing mean:", inc1.mean())
    print("Decreasing mean:", dec1.mean())
    print("Difference:", inc1.mean() - dec1.mean())
    print("T-test:", ttest_ind(inc1, dec1, equal_var=False))
    print("Cohen's d:", cohens_d(inc1, dec1))

    tmp1 = dataset[["EF1plus_change", var]].dropna()

    print(
        "Pearson:",
        pearsonr(tmp1["EF1plus_change"], tmp1[var])
    )
    print(
        "Spearman:",
        spearmanr(tmp1["EF1plus_change"], tmp1[var])
    )

    # F/EF2+
    inc2 = dataset.loc[
        dataset["TREND_GROUP_EF2"] == "Increasing",
        var
    ].dropna()

    dec2 = dataset.loc[
        dataset["TREND_GROUP_EF2"] == "Decreasing",
        var
    ].dropna()

    print("\nF/EF2+")
    print("Increasing mean:", inc2.mean())
    print("Decreasing mean:", dec2.mean())
    print("Difference:", inc2.mean() - dec2.mean())
    print("T-test:", ttest_ind(inc2, dec2, equal_var=False))
    print("Cohen's d:", cohens_d(inc2, dec2))

    tmp2 = dataset[["EF2plus_change", var]].dropna()

    print(
        "Pearson:",
        pearsonr(tmp2["EF2plus_change"], tmp2[var])
    )
    print(
        "Spearman:",
        spearmanr(tmp2["EF2plus_change"], tmp2[var])
    )

test_variable("AGE_65_PLUS_RATE")
test_variable("UNDER_5_RATE")

## Emergency hospital access

In [ ]:
hospitals = pd.read_csv(RAW_FILES_DIR / "Hospital_General_Information.csv")

In [ ]:
hospitals.columns.tolist()

In [ ]:
print(hospitals[["State", "County/Parish", "Emergency Services"]].head(20))

print("\nEmergency Services values:")
print(hospitals["Emergency Services"].value_counts(dropna=False))

print("\nHospital types:")
print(hospitals["Hospital Type"].value_counts(dropna=False))

In [ ]:
import pandas as pd
import numpy as np

# State abbreviation -> full state name
state_map = {
    "AL":"ALABAMA","AK":"ALASKA","AZ":"ARIZONA","AR":"ARKANSAS",
    "CA":"CALIFORNIA","CO":"COLORADO","CT":"CONNECTICUT","DE":"DELAWARE",
    "FL":"FLORIDA","GA":"GEORGIA","HI":"HAWAII","ID":"IDAHO",
    "IL":"ILLINOIS","IN":"INDIANA","IA":"IOWA","KS":"KANSAS",
    "KY":"KENTUCKY","LA":"LOUISIANA","ME":"MAINE","MD":"MARYLAND",
    "MA":"MASSACHUSETTS","MI":"MICHIGAN","MN":"MINNESOTA","MS":"MISSISSIPPI",
    "MO":"MISSOURI","MT":"MONTANA","NE":"NEBRASKA","NV":"NEVADA",
    "NH":"NEW HAMPSHIRE","NJ":"NEW JERSEY","NM":"NEW MEXICO","NY":"NEW YORK",
    "NC":"NORTH CAROLINA","ND":"NORTH DAKOTA","OH":"OHIO","OK":"OKLAHOMA",
    "OR":"OREGON","PA":"PENNSYLVANIA","RI":"RHODE ISLAND","SC":"SOUTH CAROLINA",
    "SD":"SOUTH DAKOTA","TN":"TENNESSEE","TX":"TEXAS","UT":"UTAH",
    "VT":"VERMONT","VA":"VIRGINIA","WA":"WASHINGTON","WV":"WEST VIRGINIA",
    "WI":"WISCONSIN","WY":"WYOMING","DC":"DISTRICT OF COLUMBIA"
}

# Clean hospital data
hosp = hospitals[
    ["State", "County/Parish", "Emergency Services"]
].copy()

hosp["STATE"] = hosp["State"].map(state_map)

# Normalize county names
hosp["COUNTY_KEY"] = (
    hosp["County/Parish"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(" COUNTY", "", regex=False)
    .str.replace(" PARISH", "", regex=False)
)

# Aggregate hospital counts
hospital_county = (
    hosp.groupby(["STATE", "COUNTY_KEY"])
    .agg(
        HOSPITAL_COUNT=("Emergency Services", "size"),
        EMERGENCY_HOSPITAL_COUNT=(
            "Emergency Services",
            lambda x: (x == "Yes").sum()
        )
    )
    .reset_index()
)

hospital_county["HAS_EMERGENCY_HOSPITAL"] = (
    hospital_county["EMERGENCY_HOSPITAL_COUNT"] > 0
).astype(int)

# Create matching county key in tornado dataset
dataset["COUNTY_KEY"] = (
    dataset["COUNTY_NAME"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(" COUNTY", "", regex=False)
    .str.replace(" PARISH", "", regex=False)
)

# Merge
dataset = dataset.merge(
    hospital_county,
    on=["STATE", "COUNTY_KEY"],
    how="left"
)

# Counties with no CMS hospital record get zero
dataset[
    ["HOSPITAL_COUNT", "EMERGENCY_HOSPITAL_COUNT", "HAS_EMERGENCY_HOSPITAL"]
] = dataset[
    ["HOSPITAL_COUNT", "EMERGENCY_HOSPITAL_COUNT", "HAS_EMERGENCY_HOSPITAL"]
].fillna(0)

print("Missing hospital count:", dataset["HOSPITAL_COUNT"].isna().sum())
print("Counties with zero hospitals:", (dataset["HOSPITAL_COUNT"] == 0).sum())
print("Counties with zero emergency hospitals:",
      (dataset["EMERGENCY_HOSPITAL_COUNT"] == 0).sum())

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

# Binary vulnerability variable
dataset["NO_EMERGENCY_HOSPITAL"] = (
    dataset["EMERGENCY_HOSPITAL_COUNT"] == 0
).astype(int)

# Compare increasing vs decreasing F/EF1+
ef1 = dataset[
    dataset["TREND_GROUP"].isin(["Increasing", "Decreasing"])
].copy()

ef1_table = pd.crosstab(
    ef1["TREND_GROUP"],
    ef1["NO_EMERGENCY_HOSPITAL"]
)

chi2_ef1, p_ef1, dof_ef1, expected_ef1 = chi2_contingency(ef1_table)

ef1_rates = (
    ef1.groupby("TREND_GROUP")["NO_EMERGENCY_HOSPITAL"]
    .mean() * 100
)

print("===== F/EF1+ =====")
print("Percent with no emergency hospital:")
print(ef1_rates)
print("\nChi-square:", chi2_ef1)
print("p-value:", p_ef1)


# Compare increasing vs decreasing F/EF2+
ef2 = dataset[
    dataset["TREND_GROUP_EF2"].isin(["Increasing", "Decreasing"])
].copy()

ef2_table = pd.crosstab(
    ef2["TREND_GROUP_EF2"],
    ef2["NO_EMERGENCY_HOSPITAL"]
)

chi2_ef2, p_ef2, dof_ef2, expected_ef2 = chi2_contingency(ef2_table)

ef2_rates = (
    ef2.groupby("TREND_GROUP_EF2")["NO_EMERGENCY_HOSPITAL"]
    .mean() * 100
)

print("\n===== F/EF2+ =====")
print("Percent with no emergency hospital:")
print(ef2_rates)
print("\nChi-square:", chi2_ef2)
print("p-value:", p_ef2)

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np
import matplotlib.pyplot as plt

results = []

for label, group_col in [
    ("F/EF1+", "TREND_GROUP"),
    ("F/EF2+", "TREND_GROUP_EF2")
]:
    temp = dataset[
        dataset[group_col].isin(["Increasing", "Decreasing"])
    ].copy()

    table = pd.crosstab(
        temp[group_col],
        temp["NO_EMERGENCY_HOSPITAL"]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    rates = (
        temp.groupby(group_col)["NO_EMERGENCY_HOSPITAL"]
        .mean() * 100
    )

    decreasing = rates["Decreasing"]
    increasing = rates["Increasing"]

    # Relative risk
    relative_risk = increasing / decreasing

    # Cramer's V
    n = table.to_numpy().sum()
    cramers_v = np.sqrt(chi2 / n)

    results.append({
        "threshold": label,
        "decreasing": decreasing,
        "increasing": increasing,
        "difference": increasing - decreasing,
        "relative_risk": relative_risk,
        "p_value": p,
        "cramers_v": cramers_v
    })

for r in results:
    print(f"\n===== {r['threshold']} =====")
    print(f"Decreasing: {r['decreasing']:.2f}%")
    print(f"Increasing: {r['increasing']:.2f}%")
    print(f"Difference: +{r['difference']:.2f} percentage points")
    print(f"Relative risk: {r['relative_risk']:.3f}x")
    print(f"p-value: {r['p_value']:.6g}")
    print(f"Cramer's V: {r['cramers_v']:.3f}")

# Chart
labels = ["F/EF1+", "F/EF2+"]
decreasing = [r["decreasing"] for r in results]
increasing = [r["increasing"] for r in results]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8, 6))

plt.bar(x - width/2, decreasing, width, label="Decreasing")
plt.bar(x + width/2, increasing, width, label="Increasing")

plt.xticks(x, labels)
plt.ylabel("Counties with no emergency hospital (%)")
plt.xlabel("Tornado intensity threshold")
plt.title("Emergency Hospital Availability by Tornado Activity Trend")
plt.legend()

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Binary indicators: 1 = tornado activity increased, 0 = decreased
dataset["EF1_INCREASING"] = np.where(
    dataset["TREND_GROUP"] == "Increasing", 1,
    np.where(dataset["TREND_GROUP"] == "Decreasing", 0, np.nan)
)

dataset["EF2_INCREASING"] = np.where(
    dataset["TREND_GROUP_EF2"] == "Increasing", 1,
    np.where(dataset["TREND_GROUP_EF2"] == "Decreasing", 0, np.nan)
)

# ---------- F/EF1+ ----------
ef1_logit = dataset[
    [
        "NO_EMERGENCY_HOSPITAL",
        "EF1_INCREASING",
        "mean_lat",
        "mean_lon"
    ]
].dropna()

X1 = sm.add_constant(
    ef1_logit[
        ["EF1_INCREASING", "mean_lat", "mean_lon"]
    ]
)

y1 = ef1_logit["NO_EMERGENCY_HOSPITAL"]

model_ef1 = sm.Logit(y1, X1).fit()

print("\n===== F/EF1+ LOGISTIC REGRESSION =====")
print(model_ef1.summary())

print("\nOdds Ratios:")
print(np.exp(model_ef1.params))


# ---------- F/EF2+ ----------
ef2_logit = dataset[
    [
        "NO_EMERGENCY_HOSPITAL",
        "EF2_INCREASING",
        "mean_lat",
        "mean_lon"
    ]
].dropna()

X2 = sm.add_constant(
    ef2_logit[
        ["EF2_INCREASING", "mean_lat", "mean_lon"]
    ]
)

y2 = ef2_logit["NO_EMERGENCY_HOSPITAL"]

model_ef2 = sm.Logit(y2, X2).fit()

print("\n===== F/EF2+ LOGISTIC REGRESSION =====")
print(model_ef2.summary())

print("\nOdds Ratios:")
print(np.exp(model_ef2.params))

## Rural status and adjusted hospital access models

In [ ]:
rural = pd.read_csv(RAW_FILES_DIR / "Ruralurbancontinuumcodes2023.csv", encoding="latin1")

In [ ]:
print(rural.shape)
print(rural.columns.tolist())
rural.head()

In [ ]:
import pandas as pd
import numpy as np

# Keep only population and RUCC rows
rural_clean = rural[
    rural["Attribute"].isin(["Population_2020", "RUCC_2023"])
].copy()

# Pivot so each county is one row
rural_clean = (
    rural_clean
    .pivot_table(
        index=["FIPS", "State", "County_Name"],
        columns="Attribute",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

# Clean FIPS
rural_clean["COUNTY_FIPS"] = (
    rural_clean["FIPS"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

# Convert numeric columns
rural_clean["Population_2020"] = pd.to_numeric(
    rural_clean["Population_2020"],
    errors="coerce"
)

rural_clean["RUCC_2023"] = pd.to_numeric(
    rural_clean["RUCC_2023"],
    errors="coerce"
)

# RUCC 1-3 = metro, 4-9 = nonmetro
rural_clean["NONMETRO"] = (
    rural_clean["RUCC_2023"] >= 4
).astype(int)

# Merge into main dataset
dataset = dataset.merge(
    rural_clean[
        ["COUNTY_FIPS", "Population_2020", "RUCC_2023", "NONMETRO"]
    ],
    on="COUNTY_FIPS",
    how="left"
)

print("Missing RUCC:", dataset["RUCC_2023"].isna().sum())
print("Missing population:", dataset["Population_2020"].isna().sum())

print("\nRUCC distribution:")
print(dataset["RUCC_2023"].value_counts().sort_index())

print("\nMetro vs nonmetro:")
print(dataset["NONMETRO"].value_counts())

In [ ]:
import numpy as np
import statsmodels.api as sm

# =========================
# F/EF1+ logistic regression
# =========================

ef1 = dataset[
    [
        "NO_EMERGENCY_HOSPITAL",
        "EF1_INCREASING",
        "mean_lat",
        "mean_lon",
        "NONMETRO"
    ]
].dropna()

X1 = sm.add_constant(
    ef1[
        [
            "EF1_INCREASING",
            "mean_lat",
            "mean_lon",
            "NONMETRO"
        ]
    ]
)

y1 = ef1["NO_EMERGENCY_HOSPITAL"]

model1 = sm.Logit(y1, X1).fit()

print("\n===== F/EF1+ LOGISTIC REGRESSION =====")
print(model1.summary())

print("\nOdds Ratios:")
print(np.exp(model1.params))


# =========================
# F/EF2+ logistic regression
# =========================

ef2 = dataset[
    [
        "NO_EMERGENCY_HOSPITAL",
        "EF2_INCREASING",
        "mean_lat",
        "mean_lon",
        "NONMETRO"
    ]
].dropna()

X2 = sm.add_constant(
    ef2[
        [
            "EF2_INCREASING",
            "mean_lat",
            "mean_lon",
            "NONMETRO"
        ]
    ]
)

y2 = ef2["NO_EMERGENCY_HOSPITAL"]

model2 = sm.Logit(y2, X2).fit()

print("\n===== F/EF2+ LOGISTIC REGRESSION =====")
print(model2.summary())

print("\nOdds Ratios:")
print(np.exp(model2.params))

## Regression plots for statistically significant associations

This section runs independently on the saved `data/vulnerability_analysis/dataset.csv` (not the in-memory dataset). It tests seven continuous vulnerability measures against each tornado-change threshold using unadjusted OLS with HC3 robust standard errors, plus two hospital-access logistic models adjusted for latitude, longitude, and nonmetro status. Counties with missing model inputs are excluded separately for each fit.

The full table reports all 16 tests. Only associations with Benjamini–Hochberg **FDR q < 0.05** are plotted. Continuous plots show a fitted line and pointwise 95% confidence interval for the mean; hospital plots show adjusted odds ratios and 95% confidence intervals. These intervals are not multiplicity-adjusted. Continuous slopes describe a one-event change; hospital odds ratios compare increasing with decreasing activity, excluding no-change counties.

These are exploratory associations, not causal effects. Robust standard errors do not account for spatial dependence between counties. The continuous regressions are new analyses, distinct from the earlier group t-tests and rank correlations. PNG/PDF figures and the full results table are saved under `figures/regressions`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display

# This section can run independently using the saved county dataset.
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / 'data' / 'vulnerability_analysis' / 'dataset.csv').is_file():
        regression_root = directory
        break
else:
    raise FileNotFoundError('Cannot locate data/vulnerability_analysis/dataset.csv')
regression_data = pd.read_csv(
    regression_root / 'data' / 'vulnerability_analysis' / 'dataset.csv',
    dtype={'COUNTY_FIPS': 'string'},
)
if regression_data['COUNTY_FIPS'].duplicated().any():
    raise ValueError('Regression input must contain one row per county.')
regression_output = regression_root / 'figures' / 'regressions'
regression_output.mkdir(parents=True, exist_ok=True)

outcomes = {
    'POVERTY_RATE': 'Poverty rate (%)',
    'MEDIAN_HOUSEHOLD_INCOME': 'Median household income ($)',
    'PERCENT_UNINSURED': 'Uninsured (%)',
    'NO_VEHICLE_RATE': 'Households without a vehicle (%)',
    'DISABILITY_RATE': 'Disability prevalence (%)',
    'UNDER_5_RATE': 'Population under age 5 (%)',
    'AGE_65_PLUS_RATE': 'Population age 65+ (%)',
}
regression_fits = {}
regression_rows = []
for threshold in ['EF1', 'EF2']:
    predictor = f'{threshold}plus_change'
    for outcome, label in outcomes.items():
        sample = regression_data[[predictor, outcome]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(sample) < 3 or sample[predictor].nunique() < 2:
            raise ValueError(f'Insufficient data for {threshold}: {outcome}')
        fit = sm.OLS(sample[outcome], sm.add_constant(sample[[predictor]])).fit(cov_type='HC3')
        key = f'{threshold}_{outcome}'
        regression_fits[key] = (fit, sample, predictor, outcome)
        lo, hi = fit.conf_int().loc[predictor]
        regression_rows.append(dict(
            key=key, threshold=threshold, outcome=outcome, model='OLS (HC3)',
            n=int(fit.nobs), estimate=fit.params[predictor], ci_low=lo, ci_high=hi,
            p_value=fit.pvalues[predictor], r_squared=fit.rsquared,
        ))

# Retain the notebook's hospital model: increasing vs decreasing activity,
# adjusted for county latitude, longitude, and nonmetro status.
for threshold in ['EF1', 'EF2']:
    predictor = f'{threshold}_INCREASING'
    outcome = 'NO_EMERGENCY_HOSPITAL'
    columns = [predictor, 'mean_lat', 'mean_lon', 'NONMETRO']
    sample = regression_data[columns + [outcome]].replace([np.inf, -np.inf], np.nan).dropna()
    fit = sm.Logit(sample[outcome], sm.add_constant(sample[columns])).fit(disp=False, cov_type='HC3')
    if not fit.mle_retvals['converged']:
        raise RuntimeError(f'{threshold} hospital model did not converge')
    key = f'{threshold}_{outcome}'
    regression_fits[key] = (fit, sample, predictor, outcome)
    lo, hi = np.exp(fit.conf_int().loc[predictor])
    regression_rows.append(dict(
        key=key, threshold=threshold, outcome=outcome, model='Adjusted logistic (HC3)',
        n=int(fit.nobs), estimate=np.exp(fit.params[predictor]), ci_low=lo, ci_high=hi,
        p_value=fit.pvalues[predictor], r_squared=np.nan,
    ))

regression_results = pd.DataFrame(regression_rows)
regression_results['q_value'] = multipletests(regression_results['p_value'], method='fdr_bh')[1]
regression_results['significant'] = regression_results['q_value'] < 0.05
regression_results = regression_results.sort_values('q_value').reset_index(drop=True)
regression_results.to_csv(regression_output / 'regression_results.csv', index=False)
display(regression_results.drop(columns='key'))
print(f"{regression_results['significant'].sum()} of {len(regression_results)} associations have FDR q < 0.05.")


### Hospital access: adjusted probabilities

The hospital outcome is binary, so logistic regression is appropriate. These plots compare increasing versus decreasing tornado activity, adjusting for latitude, longitude, and nonmetro status. Both groups are standardized over the same model sample's observed covariates; counties with no change or missing inputs are excluded. Error bars are pointwise 95% delta-method confidence intervals using HC3 covariance, conditional on the observed covariates. The p-values test the tornado-trend coefficient, and q-values use the 16-test FDR correction above. These associations do not establish causation.

Observed percentages and adjusted probabilities are both included in the table. Odds ratios describe odds, not percentage increases in probability. The outcome retains the existing dataset's hospital matching and zero-count classification; it does not measure travel time or access to hospitals across county borders.

In [ ]:
from scipy.special import expit, logit

hospital_predictions = []
fig, axes = plt.subplots(1, 2, figsize=(11, 5.8), sharey=True, layout='constrained')
for ax, threshold in zip(axes, ['EF1', 'EF2']):
    key = f'{threshold}_NO_EMERGENCY_HOSPITAL'
    fit, sample, predictor, outcome = regression_fits[key]
    result = regression_results.set_index('key').loc[key]
    label = 'F/EF1+' if threshold == 'EF1' else 'F/EF2+'
    for group, group_name, color in [(0, 'Decreasing', '#436b85'), (1, 'Increasing', '#b84630')]:
        # Standardize both groups over the same observed covariate distribution.
        design = pd.DataFrame(fit.model.exog.copy(), columns=fit.model.exog_names)
        design[predictor] = group
        probabilities = expit(design.to_numpy() @ fit.params.to_numpy())
        probability = probabilities.mean()
        gradient = (design.to_numpy() * (probabilities * (1 - probabilities))[:, None]).mean(axis=0)
        variance = gradient @ fit.cov_params().to_numpy() @ gradient
        # Delta-method CI on the logit scale keeps probability bounds within [0, 1].
        logit_se = np.sqrt(max(variance, 0)) / (probability * (1 - probability))
        lo, hi = expit(logit(probability) + np.array([-1, 1]) * 1.96 * logit_se)
        observed = sample.loc[sample[predictor] == group, outcome].mean()
        hospital_predictions.append(dict(
            threshold=label, trend=group_name, n=int(fit.nobs),
            observed_percent=100*observed, adjusted_percent=100*probability,
            ci_low_percent=100*lo, ci_high_percent=100*hi,
            odds_ratio=result['estimate'], p_value=result['p_value'], q_value=result['q_value'],
        ))
        ax.errorbar(group, 100*probability,
                    yerr=[[100*(probability-lo)], [100*(hi-probability)]],
                    fmt='o', color=color, capsize=7, markersize=9, linewidth=2)
        ax.annotate(f'{100*probability:.1f}%', (group, 100*hi),
                    xytext=(0, 9), textcoords='offset points', ha='center', fontsize=12, color=color)
    ax.set_xticks([0, 1], ['Decreasing', 'Increasing'])
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(0, 100)
    ax.set_xlabel(f'{label} tornado activity')
    ax.set_title(f'{label} | {int(fit.nobs):,} counties\n'
                 f'Adjusted OR {result["estimate"]:.2f} '
                 f'(95% CI {result["ci_low"]:.2f}–{result["ci_high"]:.2f})\n'
                 f'p={result["p_value"]:.3g}; FDR q={result["q_value"]:.3g}', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.15)
axes[0].set_ylabel('Adjusted probability of no emergency hospital (%)')
fig.suptitle('Emergency hospital access by tornado activity trend\n'
             'Adjusted for latitude, longitude, and nonmetro status · 95% confidence intervals', fontsize=13)
for extension in ['png', 'pdf']:
    fig.savefig(regression_output / f'hospital_adjusted_probabilities.{extension}', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
hospital_predictions = pd.DataFrame(hospital_predictions)
hospital_predictions.to_csv(regression_output / 'hospital_adjusted_probabilities.csv', index=False)
display(hospital_predictions)


### All significant regression results


In [ ]:
regression_artifacts = []
for row in regression_results.loc[regression_results['significant']].itertuples():
    fit, sample, predictor, outcome = regression_fits[row.key]
    fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
    threshold_label = 'F/EF1+' if row.threshold == 'EF1' else 'F/EF2+'
    if row.model.startswith('OLS'):
        grid = np.linspace(sample[predictor].min(), sample[predictor].max(), 200)
        prediction = fit.get_prediction(sm.add_constant(pd.DataFrame({predictor: grid}))).summary_frame()
        ax.scatter(sample[predictor], sample[outcome], s=12, alpha=0.18, color='#436b85', edgecolors='none', rasterized=True)
        ax.plot(grid, prediction['mean'], color='#b84630', linewidth=2, label='OLS fit')
        ax.fill_between(grid, prediction['mean_ci_lower'], prediction['mean_ci_upper'], color='#b84630', alpha=0.2, label='95% pointwise mean CI (HC3)')
        ax.set_xlabel(f'{threshold_label} event-count change: 1988–2025 minus 1950–1987')
        ax.set_ylabel(outcomes[outcome])
        ax.set_title(f'{outcomes[outcome]} vs. {threshold_label} tornado change\n'
                     f'n={row.n:,} | slope={row.estimate:.3g} | R²={row.r_squared:.3f}\n'
                     f'p={row.p_value:.3g} | FDR q={row.q_value:.3g}', fontsize=11)
        ax.legend(frameon=False, fontsize=9)
    else:
        ax.errorbar(row.estimate, 0, xerr=[[row.estimate-row.ci_low], [row.ci_high-row.estimate]],
                    fmt='o', capsize=7, color='#b84630', markersize=8)
        ax.axvline(1, linestyle='--', color='gray', label='No association (OR = 1)')
        ax.set_xscale('log')
        ax.set_yticks([0], ['Increasing vs. decreasing'])
        ax.set_ylim(-0.8, 0.8)
        ax.set_xlabel('Adjusted odds ratio for no emergency hospital (95% CI)')
        ax.set_title(f'{threshold_label} tornado trend and hospital access\n'
                     f'OR={row.estimate:.2f} [{row.ci_low:.2f}, {row.ci_high:.2f}] | n={row.n:,}\n'
                     f'p={row.p_value:.3g} | FDR q={row.q_value:.3g}', fontsize=11)
        ax.legend(frameon=False, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    for extension in ['png', 'pdf']:
        destination = regression_output / f'{row.key}.{extension}'
        fig.savefig(destination, dpi=200, bbox_inches='tight')
        regression_artifacts.append(str(destination.relative_to(regression_root)))
    plt.show()
    plt.close(fig)
if not regression_results['significant'].any():
    print('No associations meet the FDR threshold; no regression plots generated.')
print(f'Figures and the complete results table saved to {regression_output}')
